# 3RScan Temporal CLIP Embedding Dataset Builder

This notebook builds training data for your residual model by:

1. Rendering instance masks from the segmented mesh (`labels.instances.annotated.v2.ply`) for each RGB frame pose.
2. Aligning rendered instance masks with original RGB images.
3. Extracting masked object crops.
4. Computing CLIP embeddings for each visible object per frame.
5. Saving per-scan JSON files grouped by `scan_id -> object_id -> global_id`.

The code is designed to run on many scans (for example 600 scans) with checkpoint-friendly per-scan outputs.

In [1]:
import io
import os
import re
import json
import math
import shutil
import tempfile
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import clip

from pytorch3d.io import IO
from pytorch3d.structures import Meshes
from pytorch3d.renderer import (
    RasterizationSettings,
    MeshRasterizer,
    TexturesVertex,
)
from pytorch3d.renderer.cameras import PerspectiveCameras
from pytorch3d.ops import interpolate_face_attributes


@dataclass
class BuildConfig:
    dataset_root: Path
    objects_json_path: Path
    output_root: Path
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    renderer_backend: str = "pyrender"  # "pyrender" (OpenGL offscreen) or "pytorch3d"
    renderer_device: str = "cpu"
    clip_model_name: str = "ViT-B/32"
    clip_batch_size: int = 128
    min_visible_pixels: int = 64
    frame_stride: int = 1
    max_scans: Optional[int] = None
    save_overlay_debug: bool = False
    save_comparison_debug: bool = False
    comparison_every_n_frames: int = 100
    overlay_alpha: float = 0.45
    use_nearest_color_fallback: bool = False


print(f"Using CLIP device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("Renderer backend default: pyrender")

Using CLIP device: cuda
Renderer backend default: pyrender


In [3]:
def parse_info_txt(raw_text: str) -> dict:
    info = {}
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or "=" not in line:
            continue
        k, v = line.split("=", 1)
        info[k.strip()] = v.strip()
    return info


def parse_mat4_from_text(raw_text: str) -> np.ndarray:
    vals = [float(x) for x in raw_text.replace("\n", " ").split()]
    if len(vals) != 16:
        raise ValueError(f"Expected 16 values for pose, got {len(vals)}")
    return np.asarray(vals, dtype=np.float32).reshape(4, 4)


def parse_hex_color(hex_color: str) -> Tuple[int, int, int]:
    h = hex_color.strip().lstrip("#")
    if len(h) != 6:
        raise ValueError(f"Invalid color: {hex_color}")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def rgb_to_color_key(rgb: Tuple[int, int, int]) -> int:
    r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])
    return (r << 16) | (g << 8) | b


def color_key_to_rgb(color_key: int) -> Tuple[int, int, int]:
    return ((color_key >> 16) & 255, (color_key >> 8) & 255, color_key & 255)


def rgb_image_to_color_key_image(rgb_img: np.ndarray) -> np.ndarray:
    # Encode RGB triplets to one int32 channel for faster unique/mask operations.
    r = rgb_img[:, :, 0].astype(np.int32)
    g = rgb_img[:, :, 1].astype(np.int32)
    b = rgb_img[:, :, 2].astype(np.int32)
    return (r << 16) | (g << 8) | b


def load_3dssg_objects(objects_json_path: Path) -> Dict[str, dict]:
    with open(objects_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}
    scans = data.get("scans", [])
    for scan_entry in scans:
        scan_id = scan_entry["scan"]
        objects = scan_entry.get("objects", [])
        by_obj_id = {}
        by_color = {}
        by_color_key = {}
        for obj in objects:
            object_id = str(obj.get("id"))
            global_id = str(obj.get("global_id")) if obj.get("global_id") is not None else None
            label = str(obj.get("label")) if obj.get("label") is not None else "unknown"
            rgb = parse_hex_color(str(obj.get("ply_color", "#000000")))
            ckey = rgb_to_color_key(rgb)
            by_obj_id[object_id] = {
                "object_id": object_id,
                "global_id": global_id,
                "label": label,
                "ply_color": rgb,
            }
            by_color[rgb] = object_id
            by_color_key[ckey] = object_id

        out[scan_id] = {
            "by_obj_id": by_obj_id,
            "by_color": by_color,
            "by_color_key": by_color_key,
        }
    return out


def build_frame_index_from_dir(seq_dir: Path) -> List[dict]:
    color_pat = re.compile(r"frame-(\d{6})\.color\.jpg$")
    depth_pat = re.compile(r"frame-(\d{6})\.depth\.pgm$")
    pose_pat = re.compile(r"frame-(\d{6})\.pose\.txt$")

    color = {}
    depth = {}
    pose = {}

    for p in seq_dir.iterdir():
        if not p.is_file():
            continue
        n = p.name
        m = color_pat.search(n)
        if m:
            color[int(m.group(1))] = p
            continue
        m = depth_pat.search(n)
        if m:
            depth[int(m.group(1))] = p
            continue
        m = pose_pat.search(n)
        if m:
            pose[int(m.group(1))] = p

    frame_ids = sorted(set(color.keys()) & set(pose.keys()))
    return [
        {
            "frame_idx": fi,
            "color_path": color[fi],
            "depth_path": depth.get(fi),
            "pose_path": pose[fi],
        }
        for fi in frame_ids
    ]


def read_image_from_path(path: Path) -> np.ndarray:
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise RuntimeError(f"Failed to decode image {path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def read_pose_from_path(path: Path) -> np.ndarray:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    return parse_mat4_from_text(raw)


def overlay_mask(rgb: np.ndarray, mask: np.ndarray, color: Tuple[int, int, int], alpha: float = 0.45) -> np.ndarray:
    out = rgb.copy()
    c = np.asarray(color, dtype=np.float32).reshape(1, 1, 3)
    blend = (1.0 - alpha) * out.astype(np.float32) + alpha * c
    out[mask] = blend[mask].astype(np.uint8)
    return out


def make_rgb_render_comparison(rgb: np.ndarray, rendered_rgb: np.ndarray, overlay_rgb: np.ndarray) -> np.ndarray:
    h, w = rgb.shape[:2]
    if rendered_rgb.shape[:2] != (h, w):
        rendered_rgb = cv2.resize(rendered_rgb, (w, h), interpolation=cv2.INTER_NEAREST)
    if overlay_rgb.shape[:2] != (h, w):
        overlay_rgb = cv2.resize(overlay_rgb, (w, h), interpolation=cv2.INTER_NEAREST)

    panel = np.concatenate([rgb, rendered_rgb, overlay_rgb], axis=1)
    strip = np.zeros((32, panel.shape[1], 3), dtype=np.uint8)
    cv2.putText(strip, "RGB", (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(strip, "Rendered Instance", (w + 10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(strip, "Overlay", (2 * w + 10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    return np.concatenate([strip, panel], axis=0)


def mask_to_crop(rgb: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, List[int], int]:
    ys, xs = np.where(mask)
    if ys.size == 0:
        return None, None, 0
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())

    masked = np.zeros_like(rgb)
    masked[mask] = rgb[mask]
    crop = masked[y0:y1 + 1, x0:x1 + 1]
    return crop, [x0, y0, x1, y1], int(mask.sum())

In [4]:
class SegmentedMeshRenderer:
    """PyTorch3D fallback renderer."""

    def __init__(self, mesh_path: Path, device: str = "cpu"):
        io_loader = IO()
        mesh = io_loader.load_mesh(str(mesh_path))
        if mesh.isempty():
            raise RuntimeError(f"Empty mesh: {mesh_path}")

        if mesh.textures is None or not hasattr(mesh.textures, "verts_features_list"):
            raise RuntimeError(
                f"Mesh at {mesh_path} does not provide vertex colors needed for object IDs"
            )

        self.device = torch.device(device)
        verts = [v.to(self.device) for v in mesh.verts_list()]
        faces = [f.to(self.device) for f in mesh.faces_list()]
        feats = [c.to(self.device) for c in mesh.textures.verts_features_list()]
        textures = TexturesVertex(verts_features=feats)
        self.mesh = Meshes(verts=verts, faces=faces, textures=textures)

    @staticmethod
    def _twc_to_world_to_camera(t_wc: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        r_wc = t_wc[:3, :3]
        t_wc_vec = t_wc[:3, 3]
        r_cw = r_wc.T
        t_cw = -r_cw @ t_wc_vec
        return r_cw, t_cw

    def render_vertex_color_image(
        self,
        t_wc: np.ndarray,
        k: np.ndarray,
        h: int,
        w: int,
    ) -> np.ndarray:
        r_cw, t_cw = self._twc_to_world_to_camera(t_wc)

        fx, fy = float(k[0, 0]), float(k[1, 1])
        cx, cy = float(k[0, 2]), float(k[1, 2])

        cameras = PerspectiveCameras(
            focal_length=torch.tensor([[fx, fy]], dtype=torch.float32, device=self.device),
            principal_point=torch.tensor([[cx, cy]], dtype=torch.float32, device=self.device),
            image_size=torch.tensor([[h, w]], dtype=torch.float32, device=self.device),
            in_ndc=False,
            R=torch.tensor(r_cw, dtype=torch.float32, device=self.device).unsqueeze(0),
            T=torch.tensor(t_cw, dtype=torch.float32, device=self.device).unsqueeze(0),
        )

        raster_settings = RasterizationSettings(
            image_size=(h, w),
            blur_radius=0.0,
            faces_per_pixel=1,
            perspective_correct=True,
            cull_backfaces=False,
        )
        rasterizer = MeshRasterizer(cameras=cameras, raster_settings=raster_settings)
        fragments = rasterizer(self.mesh)

        pix_to_face = fragments.pix_to_face
        bary = fragments.bary_coords

        verts_rgb = self.mesh.textures.verts_features_packed()
        faces = self.mesh.faces_packed()
        face_rgb = verts_rgb[faces].unsqueeze(0)

        sampled = interpolate_face_attributes(pix_to_face, bary, face_rgb)
        rgb = sampled[0, ..., 0, :]

        valid = (pix_to_face[0, ..., 0] >= 0).unsqueeze(-1)
        rgb = torch.where(valid, rgb, torch.zeros_like(rgb))
        rgb_u8 = torch.clamp(rgb * 255.0, 0, 255).byte().cpu().numpy()
        return rgb_u8


class OpenGLSegmentedMeshRenderer:
    """Fast OpenGL offscreen renderer using pyrender."""

    def __init__(self, mesh_path: Path, width: int, height: int):
        try:
            import trimesh
            import pyrender
        except Exception as e:
            raise RuntimeError(
                "OpenGL renderer requires trimesh and pyrender. Install with: pip install trimesh pyrender"
            ) from e

        self.trimesh = trimesh
        self.pyrender = pyrender

        tm = trimesh.load(str(mesh_path), process=False)
        if isinstance(tm, trimesh.Scene):
            if len(tm.geometry) == 0:
                raise RuntimeError(f"Empty mesh scene: {mesh_path}")
            tm = trimesh.util.concatenate(tuple(tm.geometry.values()))

        if not hasattr(tm.visual, "vertex_colors") or tm.visual.vertex_colors is None:
            raise RuntimeError(f"Mesh at {mesh_path} does not provide vertex colors needed for object IDs")

        self.scene = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 0.0], ambient_light=[1.0, 1.0, 1.0, 1.0])
        self.mesh = pyrender.Mesh.from_trimesh(tm, smooth=False)
        self.mesh_node = self.scene.add(self.mesh)
        self.camera_node = None

        self.width = int(width)
        self.height = int(height)
        self.renderer = pyrender.OffscreenRenderer(viewport_width=self.width, viewport_height=self.height)

        # OpenCV camera (x right, y down, z forward) to OpenGL camera (x right, y up, z backward).
        self.cv_to_gl = np.array(
            [[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]],
            dtype=np.float32,
        )

    def _resize_if_needed(self, width: int, height: int) -> None:
        width = int(width)
        height = int(height)
        if width == self.width and height == self.height:
            return
        self.width = width
        self.height = height
        self.renderer.delete()
        self.renderer = self.pyrender.OffscreenRenderer(viewport_width=self.width, viewport_height=self.height)

    def render_vertex_color_image(self, t_wc: np.ndarray, k: np.ndarray, h: int, w: int) -> np.ndarray:
        self._resize_if_needed(w, h)

        fx, fy = float(k[0, 0]), float(k[1, 1])
        cx, cy = float(k[0, 2]), float(k[1, 2])

        cam = self.pyrender.IntrinsicsCamera(fx=fx, fy=fy, cx=cx, cy=cy, znear=0.01, zfar=1000.0)
        cam_pose_gl = (t_wc @ self.cv_to_gl).astype(np.float32)

        if self.camera_node is not None:
            self.scene.remove_node(self.camera_node)
        self.camera_node = self.scene.add(cam, pose=cam_pose_gl)

        flags = self.pyrender.constants.RenderFlags.FLAT
        color, _ = self.renderer.render(self.scene, flags=flags)

        self.scene.remove_node(self.camera_node)
        self.camera_node = None

        if color.ndim == 3 and color.shape[2] >= 3:
            return color[:, :, :3].astype(np.uint8)
        raise RuntimeError("OpenGL renderer returned invalid color buffer")

    def close(self) -> None:
        if self.renderer is not None:
            self.renderer.delete()
            self.renderer = None


class ClipImageEncoder:
    def __init__(self, model_name: str, device: str = "cuda"):
        self.device = device
        self.model, self.preprocess = clip.load(model_name, device=device)
        self.model.eval()

    @torch.no_grad()
    def encode_batch(self, image_rgb_list: List[np.ndarray], batch_size: int = 64) -> List[np.ndarray]:
        if len(image_rgb_list) == 0:
            return []

        out = []
        for i in range(0, len(image_rgb_list), batch_size):
            chunk = image_rgb_list[i:i + batch_size]
            batch = torch.stack([self.preprocess(Image.fromarray(img)) for img in chunk], dim=0).to(self.device)
            emb = self.model.encode_image(batch)
            emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            out.extend([e.detach().cpu().numpy().astype(np.float32) for e in emb])
        return out

    @torch.no_grad()
    def encode(self, image_rgb: np.ndarray) -> np.ndarray:
        return self.encode_batch([image_rgb])[0]

In [5]:
def nearest_object_id_from_color(rgb: Tuple[int, int, int], by_color: Dict[Tuple[int, int, int], str], max_dist: float = 6.0):
    if rgb in by_color:
        return by_color[rgb]

    c = np.asarray(rgb, dtype=np.float32)
    best_id = None
    best_d = 1e9
    for ref_rgb, oid in by_color.items():
        d = float(np.linalg.norm(c - np.asarray(ref_rgb, dtype=np.float32)))
        if d < best_d:
            best_d = d
            best_id = oid
    if best_d <= max_dist:
        return best_id
    return None


def build_color_to_object_lookup(scan_objects: dict):
    return dict(scan_objects.get("by_color", {})), dict(scan_objects.get("by_color_key", {}))


def ensure_scan_output_template(scan_id: str, scan_objects: dict) -> dict:
    objects_block = {}
    for oid, meta in scan_objects["by_obj_id"].items():
        objects_block[oid] = {
            "object_id": oid,
            "global_id": meta["global_id"],
            "label": meta["label"],
            "embeddings": [],
        }
    return {
        "scan_id": scan_id,
        "objects": objects_block,
    }


def process_single_scan(
    scan_dir: Path,
    scan_objects_map: Dict[str, dict],
    cfg: BuildConfig,
    encoder: ClipImageEncoder,
) -> Optional[Path]:
    scan_id = scan_dir.name
    if scan_id not in scan_objects_map:
        print(f"[SKIP] {scan_id}: not found in 3DSSG objects metadata")
        return None

    sequence_zip = scan_dir / "sequence.zip"
    sequence_dir = scan_dir / "sequence"
    labels_ply = scan_dir / "labels.instances.annotated.v2.ply"

    if not labels_ply.exists():
        print(f"[SKIP] {scan_id}: missing labels.instances.annotated.v2.ply")
        return None

    has_sequence_dir = sequence_dir.exists() and sequence_dir.is_dir()
    has_sequence_zip = sequence_zip.exists()

    if not has_sequence_dir and not has_sequence_zip:
        print(f"[SKIP] {scan_id}: missing both sequence/ and sequence.zip")
        return None

    scan_objects = scan_objects_map[scan_id]
    by_color, by_color_key = build_color_to_object_lookup(scan_objects)
    result = ensure_scan_output_template(scan_id, scan_objects)

    debug_dir = cfg.output_root / "debug_overlays" / scan_id
    comp_dir = cfg.output_root / "debug_comparison" / scan_id
    tmp_root = cfg.output_root / "_tmp_sequences"
    tmp_root.mkdir(parents=True, exist_ok=True)

    if cfg.save_overlay_debug:
        debug_dir.mkdir(parents=True, exist_ok=True)
    if cfg.save_comparison_debug:
        comp_dir.mkdir(parents=True, exist_ok=True)

    def run_on_sequence_dir(seq_root: Path) -> None:
        info_path = seq_root / "_info.txt"
        if not info_path.exists():
            print(f"[SKIP] {scan_id}: sequence folder has no _info.txt")
            return

        with open(info_path, "r", encoding="utf-8") as f:
            info = parse_info_txt(f.read())

        color_w = int(info["m_colorWidth"])
        color_h = int(info["m_colorHeight"])
        k_color_vals = [float(x) for x in info["m_calibrationColorIntrinsic"].split()]
        k_color = np.asarray(k_color_vals, dtype=np.float32).reshape(4, 4)[:3, :3]

        backend = str(cfg.renderer_backend).lower()
        if backend in ("pyrender", "opengl"):
            renderer = OpenGLSegmentedMeshRenderer(labels_ply, width=color_w, height=color_h)
        else:
            renderer = SegmentedMeshRenderer(labels_ply, device=cfg.renderer_device)

        frames = build_frame_index_from_dir(seq_root)
        if cfg.frame_stride > 1:
            frames = frames[:: cfg.frame_stride]

        render_fail_count = 0

        try:
            for fr in tqdm(frames, desc=f"scan {scan_id}", leave=False):
                frame_idx = int(fr["frame_idx"])
                rgb = read_image_from_path(fr["color_path"])
                t_wc = read_pose_from_path(fr["pose_path"])

                if not np.isfinite(t_wc).all():
                    continue

                try:
                    rendered_obj_rgb = renderer.render_vertex_color_image(
                        t_wc=t_wc,
                        k=k_color,
                        h=color_h,
                        w=color_w,
                    )
                except Exception as e:
                    render_fail_count += 1
                    if render_fail_count <= 5:
                        print(f"[WARN] {scan_id} frame {frame_idx}: render failed: {e}")
                    continue

                if rendered_obj_rgb.shape[:2] != rgb.shape[:2]:
                    rendered_obj_rgb = cv2.resize(
                        rendered_obj_rgb,
                        (rgb.shape[1], rgb.shape[0]),
                        interpolation=cv2.INTER_NEAREST,
                    )

                # Fast path: encode RGB colors to one int32 channel.
                color_key_img = rgb_image_to_color_key_image(rendered_obj_rgb)
                uniq_keys, uniq_counts = np.unique(color_key_img, return_counts=True)

                frame_objects = []
                overlay_img = rgb.copy()

                for ckey, count in zip(uniq_keys.tolist(), uniq_counts.tolist()):
                    ckey = int(ckey)
                    if ckey == 0:
                        continue
                    if count < int(cfg.min_visible_pixels):
                        continue

                    oid = by_color_key.get(ckey)
                    if oid is None and cfg.use_nearest_color_fallback:
                        rgb_tuple = color_key_to_rgb(ckey)
                        oid = nearest_object_id_from_color(rgb_tuple, by_color)
                    if oid is None:
                        continue

                    mask = color_key_img == ckey
                    crop, bbox, px = mask_to_crop(rgb, mask)
                    if crop is None or crop.size == 0:
                        continue

                    frame_objects.append(
                        {
                            "object_id": oid,
                            "color_key": ckey,
                            "mask": mask,
                            "crop": crop,
                            "bbox": bbox,
                            "visible_pixels": px,
                            "frame_name": fr["color_path"].name,
                            "frame_idx": frame_idx,
                        }
                    )

                    if cfg.save_overlay_debug or cfg.save_comparison_debug:
                        overlay_img = overlay_mask(
                            overlay_img,
                            mask,
                            color_key_to_rgb(ckey),
                            alpha=cfg.overlay_alpha,
                        )

                if len(frame_objects) > 0:
                    batch_crops = [item["crop"] for item in frame_objects]
                    batch_embs = encoder.encode_batch(batch_crops, batch_size=int(cfg.clip_batch_size))

                    for item, emb in zip(frame_objects, batch_embs):
                        obj_block = result["objects"][item["object_id"]]
                        obj_block["embeddings"].append(
                            {
                                "frame_index": item["frame_idx"],
                                "frame_name": item["frame_name"],
                                "bbox_xyxy": item["bbox"],
                                "visible_pixels": item["visible_pixels"],
                                "embedding": emb.tolist(),
                            }
                        )

                if cfg.save_overlay_debug:
                    out_dbg = debug_dir / f"frame-{frame_idx:06d}.png"
                    Image.fromarray(overlay_img).save(out_dbg)

                if cfg.save_comparison_debug and (frame_idx % max(1, int(cfg.comparison_every_n_frames)) == 0):
                    comparison = make_rgb_render_comparison(rgb, rendered_obj_rgb, overlay_img)
                    comp_path = comp_dir / f"frame-{frame_idx:06d}.png"
                    Image.fromarray(comparison).save(comp_path)

            if render_fail_count > 0:
                print(f"[WARN] {scan_id}: total render failures: {render_fail_count}/{len(frames)}")
        finally:
            if hasattr(renderer, "close"):
                try:
                    renderer.close()
                except Exception:
                    pass

    if has_sequence_dir:
        run_on_sequence_dir(sequence_dir)
    else:
        with tempfile.TemporaryDirectory(prefix=f"seq_{scan_id}_", dir=str(tmp_root)) as tmp_dir_str:
            tmp_dir = Path(tmp_dir_str)
            with zipfile.ZipFile(sequence_zip, "r") as zf:
                zf.extractall(tmp_dir)
            run_on_sequence_dir(tmp_dir)

    out_scans_dir = cfg.output_root / "scans"
    out_scans_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_scans_dir / f"{scan_id}.json"

    kept = {
        oid: odata
        for oid, odata in result["objects"].items()
        if len(odata["embeddings"]) > 0
    }
    result["objects"] = kept

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False)

    return out_path


def build_dataset(cfg: BuildConfig):
    cfg.output_root.mkdir(parents=True, exist_ok=True)

    scan_objects_map = load_3dssg_objects(cfg.objects_json_path)
    all_scan_dirs = [p for p in cfg.dataset_root.iterdir() if p.is_dir()]
    all_scan_dirs = sorted(all_scan_dirs, key=lambda p: p.name)

    if cfg.max_scans is not None:
        all_scan_dirs = all_scan_dirs[: cfg.max_scans]

    valid_scan_dirs = []
    skipped_missing_ply = 0
    skipped_missing_sequence = 0
    for scan_dir in all_scan_dirs:
        has_seq_dir = (scan_dir / "sequence").exists()
        has_seq_zip = (scan_dir / "sequence.zip").exists()
        if not has_seq_dir and not has_seq_zip:
            skipped_missing_sequence += 1
            continue
        if not (scan_dir / "labels.instances.annotated.v2.ply").exists():
            skipped_missing_ply += 1
            continue
        valid_scan_dirs.append(scan_dir)

    print(f"Scans discovered: {len(all_scan_dirs)}")
    print(f"Scans skipped (missing sequence folder/zip): {skipped_missing_sequence}")
    print(f"Scans skipped (missing labels.instances.annotated.v2.ply): {skipped_missing_ply}")
    print(f"Scans to process: {len(valid_scan_dirs)}")

    encoder = ClipImageEncoder(cfg.clip_model_name, device=cfg.device)

    outputs = []
    for scan_dir in tqdm(valid_scan_dirs, desc="all scans"):
        out_path = process_single_scan(
            scan_dir=scan_dir,
            scan_objects_map=scan_objects_map,
            cfg=cfg,
            encoder=encoder,
        )
        if out_path is not None:
            outputs.append(out_path)

    manifest = {
        "num_scans_discovered": len(all_scan_dirs),
        "num_scans_filtered": len(valid_scan_dirs),
        "num_skipped_missing_sequence": skipped_missing_sequence,
        "num_skipped_missing_labels_ply": skipped_missing_ply,
        "num_scans_written": len(outputs),
        "scan_files": [str(p) for p in outputs],
    }
    manifest_path = cfg.output_root / "manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    tmp_root = cfg.output_root / "_tmp_sequences"
    if tmp_root.exists():
        shutil.rmtree(tmp_root, ignore_errors=True)

    print(f"Done. Wrote {len(outputs)} scan files.")
    print(f"Manifest: {manifest_path}")
    return manifest_path

In [6]:
# Configure paths for your workspace.
cfg = BuildConfig(
    dataset_root=Path("/home/abdou/Projects/Python/RTSGS/Datasets/3rscan_full_data"),
    objects_json_path=Path("/home/abdou/Projects/Python/RTSGS/Datasets/3DSSG/3DSSG/objects.json"),
    output_root=Path("/home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset"),
    frame_stride=1,
    min_visible_pixels=128,
    max_scans=None,  # set to a small number like 2 for smoke testing
    save_overlay_debug=False,
)

print(cfg)

BuildConfig(dataset_root=PosixPath('/home/abdou/Projects/Python/RTSGS/Datasets/3rscan_full_data'), objects_json_path=PosixPath('/home/abdou/Projects/Python/RTSGS/Datasets/3DSSG/3DSSG/objects.json'), output_root=PosixPath('/home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset'), device='cuda', renderer_backend='pyrender', renderer_device='cpu', clip_model_name='ViT-B/32', clip_batch_size=128, min_visible_pixels=128, frame_stride=1, max_scans=None, save_overlay_debug=False, save_comparison_debug=False, comparison_every_n_frames=100, overlay_alpha=0.45, use_nearest_color_fallback=False)


In [7]:
# Run dataset build.
# Important: this requires labels.instances.annotated.v2.ply for each scan.
manifest_path = build_dataset(cfg)
manifest_path

Scans discovered: 601
Scans skipped (missing sequence folder/zip): 0
Scans skipped (missing labels.instances.annotated.v2.ply): 101
Scans to process: 500


all scans:   0%|          | 0/500 [00:00<?, ?it/s]

scan 05c6ede7-2e69-23b1-8b27-c1cb868f1938:   0%|          | 0/138 [00:00<?, ?it/s]

scan 09582205-e2c2-2de1-9475-1cdac7639e60:   0%|          | 0/212 [00:00<?, ?it/s]

scan 09582207-e2c2-2de1-972c-225d968c2ab4:   0%|          | 0/329 [00:00<?, ?it/s]

...

scan fcf66dbc-622d-291c-8481-6e8761c93e21:   0%|          | 0/141 [00:00<?, ?it/s]

scan fcf66dbe-622d-291c-8790-49f1fe8a02e5:   0%|          | 0/264 [00:00<?, ?it/s]

scan feefd90b-9d00-2ce5-8119-b936443cc39b:   0%|          | 0/334 [00:00<?, ?it/s]

Done. Wrote 500 scan files.
Manifest: /home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset/manifest.json


PosixPath('/home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset/manifest.json')

## If instance annotation files are missing

Your current scan folders appear to contain only:

- `mesh.refined.v2.obj`
- `mesh.refined.mtl`
- `mesh.refined_0.png`
- `sequence.zip`

To build object masks, each scan also needs:

- `labels.instances.annotated.v2.ply`
- `mesh.refined.0.010000.segs.v2.json`
- `semseg.v2.json`

You can download these by running the downloader again with the full file type set.

In [ ]:
# Optional: helper command to download missing annotation files for all scans.
# Run in terminal, not in Python notebook:
# python Datasets/download.py -o Datasets/3rscan_full_data --type labels.instances.annotated.v2.ply mesh.refined.0.010000.segs.v2.json semseg.v2.json sequence.zip mesh.refined.v2.obj mesh.refined.mtl mesh.refined_0.png
pass

In [40]:
# Optional smoke test on one scan.
# This is useful before running the full dataset build.
smoke_cfg = BuildConfig(
    dataset_root=cfg.dataset_root,
    objects_json_path=cfg.objects_json_path,
    output_root=cfg.output_root / "smoke_test",
    frame_stride=20,
    min_visible_pixels=128,
    max_scans=1,
    save_overlay_debug=True,
)

smoke_manifest = build_dataset(smoke_cfg)
smoke_manifest

Scans discovered: 1
Scans skipped (missing sequence folder/zip): 0
Scans skipped (missing labels.instances.annotated.v2.ply): 1
Scans to process: 0


all scans: 0it [00:00, ?it/s]

Done. Wrote 0 scan files.
Manifest: /home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset/smoke_test/manifest.json


PosixPath('/home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset/smoke_test/manifest.json')

In [41]:
# Quick single-scan check for the provided scan id.
single_scan_id = "1d2f850e-d757-207c-8eea-5d41656673f4"
scan_dir = cfg.dataset_root / single_scan_id

enc = ClipImageEncoder(cfg.clip_model_name, device=cfg.device)
scan_objects_map = load_3dssg_objects(cfg.objects_json_path)

single_out = process_single_scan(
    scan_dir=scan_dir,
    scan_objects_map=scan_objects_map,
    cfg=cfg,
    encoder=enc,
)

print("single_out:", single_out)
if single_out is not None and Path(single_out).exists():
    with open(single_out, "r", encoding="utf-8") as f:
        d = json.load(f)
    print("num objects with embeddings:", len(d.get("objects", {})))
    total_emb = sum(len(v.get("embeddings", [])) for v in d.get("objects", {}).values())
    print("total embeddings:", total_emb)

scan 1d2f850e-d757-207c-8eea-5d41656673f4:   0%|          | 0/153 [00:00<?, ?it/s]

single_out: /home/abdou/Projects/Python/RTSGS/SceneGraph/clip_embedding_dataset/scans/1d2f850e-d757-207c-8eea-5d41656673f4.json
num objects with embeddings: 28
total embeddings: 1080
